# 05 上下文、记忆、功能迭代与回退

## 1. 本篇对应的课程与阅读范围

上一篇已经完成“黑马记账”桌面应用。本篇处理继续开发时的四个问题：怎样保存当前代码，怎样管理越来越长的对话，怎样让项目约定跨会话保留，以及新增功能出错后怎样回到合适的状态。

按网盘中的真实顺序，对应以下四节：

| 视频 | 主题 |
|---|---|
| `10-工程化控制-git初始化和上下文压缩.mp4` | 保存代码基线，压缩上下文 |
| `11-工程化控制-memory记忆管理.mp4` | 项目约定与持久化记忆 |
| `12-工程化控制-添加新功能及git再次存档和压缩上下文.mp4` | 继续迭代，验证后再次存档 |
| `13-工程化控制-rewind命令回退代码.mp4` | 回退对话、文件与代码版本 |

主要来源是 [Part2 工程化控制和Agent开发](https://www.yuque.com/xxcls/vibecoding/ia0yu2m1v3sl537p) 中的上下文、Memory、代码回退章节，以及附录的功能开发流程；课程文件见 [VibeCoding实战网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)，提取码 `1234`。

本次已核对视频文件名、配套讲义正文和原始代码块。讲义没有完整展开视频 12 的具体新增功能、实际提交记录及逐文件修改，本篇不补造这些内容；与记账项目衔接的命令和提示词会明确标为补充。视频未逐段转录。

讲义标注适用版本为“截至 2026 年 7 月”。其中记忆路径、部分参数和回退说明存在差异，本篇就地列出依据官方文档核对后的说明，核对日期为 `2026-10-10`。

## 2. 先分清：文件、对话、约定分别保存在哪里

继续开发之前，先区分三个对象：

| 对象 | 内容 | 主要保存或管理方式 |
|---|---|---|
| 项目文件 | 源代码、配置、依赖声明、产品文档 | 本地文件与 Git 提交 |
| 当前会话上下文 | 需求讨论、读取的代码、工具结果、临时排查过程 | 当前会话、上下文压缩、会话恢复 |
| 持久约定与记忆 | 技术约束、用户偏好、长期决策、反馈 | `CLAUDE.md`、自动记忆文件 |

例如“继续用 IndexedDB + Dexie”是项目决策；“刚才 Vite 报了什么错”是当前排查信息；`src/App.tsx` 中的实现则是磁盘上的代码。

**Git 提交不会保存完整的聊天历史，压缩对话也不会替你提交代码。** 同样，恢复旧对话不意味着磁盘上的文件自动回到旧版本。这些区别决定了后面应该使用哪一种操作。

## 3. Git 初始化：让已有项目有一个代码基线

视频 10 的第一项是 Git 初始化。以下是与上一篇记账项目衔接的**操作补充**，不是视频逐字命令。

在包含 `package.json`、`CLAUDE.md`、`src/`、`src-tauri/` 的记账项目根目录打开终端，先确认当前目录和仓库状态：

```bash
git status
git rev-parse --show-toplevel
```

第二条用来确认 Git 识别的仓库根目录。若它指向上级的另一个工程，需要先明确当前项目应属于哪个仓库；不能只看见 `git status` 有输出就认为已经是记账项目自己的仓库。

如果该项目确实尚未处于 Git 仓库中，再初始化：

```bash
git init
git status
```

`git init` 创建版本管理元数据，不会把源代码自动存档，也不会自动上传到远端。已有仓库直接继续使用，无需为了本节重建。

后续所有 Git 命令都在**准备管理的记账项目仓库中**运行。本笔记中的命令是教学内容，本次整理不会对当前笔记仓库执行初始化、提交或回退。

## 4. 明确哪些文件需要进入版本管理

记账项目包含源代码，也包含安装、构建时生成的文件。保存版本之前先查看 `git status`，不要把整套依赖和编译缓存都当成业务代码。

**补充：**下面是根据上一篇实际项目结构整理的 `.gitignore` 片段，按已有文件合并，不覆盖原有规则：

```gitignore
/node_modules/
/dist/
/src-tauri/target/
/黑马记账.exe
*.tsbuildinfo
```

| 目录或文件 | 本项目中怎样处理 |
|---|---|
| `src/`、`src-tauri/src/` | 保存业务实现和桌面入口 |
| `package.json`、`Cargo.toml`、Tauri/Vite 配置 | 保存构建项目需要的声明与配置 |
| `CLAUDE.md` | 保存产品要求和项目约定 |
| `app-icon.svg`、`src-tauri/icons/` | 属于本课打包需要的图标资源，不随编译缓存一起忽略 |
| `node_modules/` | 依赖安装结果，通过依赖声明重新安装 |
| `dist/`、`src-tauri/target/`、根目录的 `黑马记账.exe` | 构建产物与 Rust 编译缓存；本例不把它们作为业务源码存档 |

`.gitignore` 主要影响尚未跟踪的文件；已经提交过的文件不会因为新写了忽略规则就自动退出版本管理。

它也不等于“AI 无法读取这些路径”。讲义将其概括为上下文过滤，可用于减少无关扫描；不能据此认为所有读取工具都会拒绝访问，或之前进入上下文的内容已经被清除。

## 5. 用 Git commit 创建检查点

Git 中需要区分三个位置：

| 位置 | 含义 |
|---|---|
| 工作区（working tree） | 当前磁盘上正在编辑的文件 |
| 暂存区（index） | 准备放进下一次提交的内容 |
| 提交（commit） | 已记录的版本快照，`HEAD` 通常指向当前提交 |

讲义给出的检查点命令是：

```bash
git add -A && git commit -m "checkpoint: before refactoring auth module"
```

这个提交说明是讲义的认证模块通用示例，不是记账项目的真实提交记录。`git add -A` 会暂存新增、修改和删除的文件；`git commit` 保存暂存区内容。

**补充：**初次为记账项目存档，可以把操作拆开，先检查再提交：

```bash
git status
git diff
git add -A
git diff --cached
git commit -m "checkpoint: accounting app baseline"
git log -1 --oneline
git status
```

`git diff` 检查未暂存改动，`git diff --cached` 检查准备提交的内容。确认这次要保存的文件都已纳入后，再创建提交。

完成后应能看到新提交。若有刻意保留的其他本地改动，工作区仍可能不干净；需要知道它们是什么，不能把它们误算成下一次 AI 修改。

如果 Git 提示缺少提交者身份，按实际身份配置对应仓库再重试，不要复制别人的姓名或邮箱。

## 6. 上下文为何需要压缩

Claude Code 的上下文包括当前对话、已读取的文件片段、工具调用结果和加载的项目说明。随着任务推进，这些内容不断累积，占用有限的上下文窗口。

讲义用“通常约 `200K tokens`”说明容量有限。具体上限取决于模型、配置与服务，不能把 `200K` 当成所有环境的固定值。

长会话的问题不只是字数多，还可能混入：

- 已放弃的技术方案。
- 修复前的旧错误与旧文件内容。
- 重复工具输出。
- 与当前功能无关的讨论。

因此管理上下文的目标是让当前任务的目标、约束、已确认决策和待办保持清楚。响应变慢或忽略早期要求可能是线索，但也可能来自网络、模型服务或工具耗时，不能仅凭速度判断上下文已满。

## 7. 使用 /compact：用摘要承接当前任务

在 **Claude Code 的对话输入框**输入，以下不是普通 Shell 命令：

```text
/compact
```

讲义将其原理描述为：整理已完成任务、关键决策、修改文件与重要事实，用摘要替代大量早期对话，以释放当前上下文空间。

课程建议在任务阶段完成、切换任务、对话变长或开始遗漏要求时主动压缩，并给出“约 30–50 轮”的经验值。这个轮数不是执行阈值；几次很大的文件读取也可能消耗大量上下文。

**补充：**当前官方文档支持在命令后说明压缩重点。针对记账项目，可以写：

```text
/compact 保留黑马记账的技术选择、已完成功能、修改文件、验证命令与结果、尚未解决的问题；重点保留 Tauri + React、IndexedDB + Dexie、Recharts、SheetJS，以及 Windows 打包曾遇到的 WiX 下载问题。
```

这是依据上一篇整理的示例，不是视频中的原始提示词。压缩后应检查摘要是否保留了仍然有效的决策，遗漏时及时补充。

### 关于“不可逆”的修正

讲义写“压缩后无法恢复原始对话”。更准确地说：**当前活动上下文不会自动保留压缩前每一轮的全部细节，摘要也可能遗漏信息。** 但不能据此认定本地会话记录必然被永久删除。当前官方文档说明，会话记录与压缩后的活动上下文不同，部分摘要操作仍保留原始会话文本供参考。

要长期保留的约定写入项目文档，要长期保留的代码创建 Git 提交，不依赖模型从摘要中重新猜出原始内容。

## 8. 减少无关读取，按任务切换上下文

讲义给出的几项做法可以直接用于后续开发：

1. 提问时限定问题、文件或函数，避免让 AI 对整个工程无限展开。
2. 大文件先搜索相关位置，再读取必要范围。
3. 已读取的文件没有变化时，避免重复完整读取。
4. 保持 `CLAUDE.md` 简短，优先写常用命令和关键约定。

讲义的精确引用示例：

```text
请查看 src/api/auth.ts:120-150 的 token 刷新逻辑
```

这是认证模块的通用示例；记账项目应换成实际存在的文件与行号。

**补充：**当前官方文档还建议在无关任务之间使用 `/clear`。需要区分：

| 操作 | 作用 | 是否恢复代码 |
|---|---|---|
| `/compact` | 压缩当前会话，保留摘要以继续任务 | 否 |
| `/clear` | 清空当前活动对话，开始新的会话上下文 | 否 |
| `/context` | 查看上下文使用情况及加载的记忆文件 | 否 |

切换前先把仍需保留的目标与约定写清楚。若是在延续同一个功能，压缩并提供当前状态通常更便于接续；若任务已经无关，可开始新会话。

## 9. Memory 管理：把长期约定留到下一次会话

课程接下来讨论记忆管理。核心问题是：新会话不应每次都重新询问同样的项目规则，同时也不能把所有临时排查内容永久保存。

当前官方文档区分两种机制：

| 机制 | 主要内容 | 谁维护 |
|---|---|---|
| `CLAUDE.md` | 明确的项目说明、约定和操作规则 | 用户维护，也可要求 AI 修改 |
| Auto memory（自动记忆） | 用户纠正、偏好、非显然的长期项目事实 | Claude 记录，用户可检查与编辑 |

二者都为模型提供上下文，不能当成必然执行的程序配置。

### 讲义路径与当前文档的差异

讲义列出 `<project>/.claude/MEMORY.md` 和 `~/.claude/MEMORY.md`。当前官方文档没有把这两个路径定义为默认的自动记忆入口。

| 当前文档中的默认位置 | 含义 |
|---|---|
| `<project>/CLAUDE.md` 或 `<project>/.claude/CLAUDE.md` | 项目说明 |
| `~/.claude/CLAUDE.md` | 适用于多个项目的用户说明 |
| `~/.claude/projects/<project>/memory/MEMORY.md` | 当前仓库的自动记忆索引 |
| 同一 `memory/` 目录下的其他 `.md` | 按主题保存的具体记忆 |

其中 `<project>` 在自动记忆路径里是 Claude Code 派生的项目标识，不是让用户直接输入这个字符串。自定义配置也可能改变存储位置。

复现时先用 `/memory` 查看实际位置，不要为了满足讲义表格自行创建一份文件，就认定 Claude 一定会自动加载。

## 10. 维护 CLAUDE.md：写约定与常用命令

讲义把 `CLAUDE.md` 称为“项目宪法”，意思是把每次开发都应知道的约定集中起来。它可以手工创建，也可以在 Claude Code 中运行：

```text
/init
```

当前官方文档说明，`/init` 会分析项目并生成初始说明；已有文件时会提出改进建议。生成后仍需检查是否与实际工程一致。

讲义中的简短结构示例：

```markdown
# 项目概述
简短描述（2-3 行）

# 常用命令
npm run dev      # 启动开发服务器
npm test         # 运行测试
npm run build    # 构建

# 关键约定
- 使用 TypeScript 严格模式
- 组件文件用 PascalCase
- API 路由放在 src/routes/
```

上面是原讲义的通用模板，不能不加核对地复制到记账项目。例如上一篇已确认 `dev`、`build`、`preview`、`tauri` 脚本，尚未确认 `npm test` 已存在；模板中的 API 路由也不代表该项目已有服务端 API。

针对记账项目，应保留已经确认的内容：Windows/macOS 目标、人民币支出、两级分类、Tauri + React、已选的存储与图表/导出库，以及“技术问题先列方案并解释”的协作约定。产品要求发生变化时同步修改，而不是让过时要求继续影响后续开发。

讲义前后分别建议 50–100 行和不超过 200 行，当前官方也建议保持精简。这些是写作建议，不是文件超过某行就一定无法加载。避免将整个 README、全部代码或一次性报错复制进去。

## 11. 查看、修改和核对记忆

在 Claude Code 中输入：

```text
/memory
```

当前官方文档说明，它可列出说明文件与记忆位置、打开文件，以及管理自动记忆。需要确认启动时哪些说明已加载，可以查看：

```text
/context
```

讲义中让 AI 记住偏好的例子是：

```text
记住我使用 pnpm 而非 npm。
```

这是演示记忆的通用例句，不是要求把上一篇的记账项目改成 pnpm。要写入某个明确的项目文件，应直接指明目标，例如“把这条项目约定加入 `CLAUDE.md`”，再检查实际修改。

讲义把记忆分为四类，当前官方文档也保留这四类：

| 类型 | 适合保存的内容 |
|---|---|
| `user` | 身份、经验、工作偏好 |
| `feedback` | 用户纠正，以及已确认应该保持的处理方式 |
| `project` | 无法直接从代码或 Git 历史推导的项目目标、约束、决策 |
| `reference` | 外部资料和信息的查找位置 |

讲义的 Tab 缩进示例使用了 `metadata.type: user`；当前官方文档说明类型写在记忆文件 frontmatter 的 `type` 字段。两者不要混成一份固定格式。本节重点是查看实际保存的文件与作用域，而不是照抄一套未经核对的目录或字段。

当前默认自动记忆索引在会话开始时只加载前 200 行或前 25KB，先达到哪项就以哪项为限；具体主题文件按需读取。这是当前文档中的行为，旧版本可能不同。

## 12. 哪些内容该记住，哪些留在当前任务中

讲义的记忆管理原则包括：不要保存从代码就能推导的信息，不保存仅对当前会话有效的临时事实，偏好变化时更新，并定期清理过时内容。

可以用记账项目理解这些区别：

| 信息 | 更合适的位置或处理 |
|---|---|
| 已决定采用两级支出分类 | 产品文档或项目约定 |
| 用户希望技术选择先解释再决定 | `CLAUDE.md` 中的协作规则 |
| 某个文件目前有多少行 | 需要时读取代码 |
| 刚才一次构建的完整日志 | 当前排查材料，必要时保留日志文件 |
| 尚未确认的故障猜测 | 留在当前任务中，确认后再决定是否沉淀 |

讲义前面建议说明项目结构，后面又说不要在记忆中复制代码可推导的信息。可按用途理解：`CLAUDE.md` 保留必要的入口与稳定约定，自动记忆不重复储存一份随代码变化的文件目录。

记忆是可修改的 Markdown。规则变化后，应更新原条目、删除矛盾表述，并检查新的会话是否加载了正确文件，不能只增加一条相反的新规则。

## 13. 继续功能迭代：先说明目标，再确认方案

视频 12 的目录显示了“添加新功能 → Git 再次存档 → 压缩上下文”的顺序。配套讲义附录的新功能流程以登录功能为例：创建工作分支与检查点、描述需求、Plan Mode 设计、确认后编码、审查和验证、提交。

本篇聚焦其中的本地迭代流程。讲义没有给出视频 12 的完整新需求，因此下面是**衔接记账项目的提示词模板**，方括号部分需要替换，不能作为已发生的课堂需求：

```text
请先阅读当前项目的 CLAUDE.md 和相关实现。

本次新增功能：[写出具体功能]。
用户操作：[用户从哪里进入，输入什么，点击什么]。
预期结果：[明确可观察的结果]。
已有约定：继续使用当前确认的技术栈与数据存储方式。

请先说明需要改哪些文件、是否影响已有账单数据、有哪些实现选择。
涉及技术取舍时，按 CLAUDE.md 的约定解释后让我决定。
完成后说明改动、验证命令和结果，以及还需要人工检查的行为。
```

课程讲义的 Plan Mode 提示是：

```text
这个任务比较复杂，请先进入 Plan Mode 设计方案。
```

先把新增行为与现有约定对齐，再进入实现。若功能涉及数据格式变化，还要明确旧账单应如何处理；不能只看新增界面是否出现，就认为迭代已完成。

## 14. 验证后再次存档，再压缩上下文

迭代完成时，需要同时检查实现和可观察行为。

| 要检查的内容 | 在记账项目中的对应证据 |
|---|---|
| 修改范围 | `git diff` 中有哪些文件和字段变化 |
| 前端可构建 | `npm run build` 的实际结果 |
| 新行为正确 | 按新增需求的操作步骤实际执行 |
| 原功能仍可用 | 原有记账、月度统计、Excel/CSV 导出仍正常 |
| 桌面发行结果 | 若交付桌面程序，重新执行打包并运行新产物 |

**补充：**在记账项目根目录执行相应检查：

```bash
git status
git diff
npm run build
```

若需要新的桌面程序，再按当前平台执行：

```bash
npx tauri build
```

`npm run build` 检查与构建前端，不能替代所有功能验证；修改源代码后继续打开旧 EXE，也不能证明新代码已生效。

讲义附录还使用 `/code-review`、`/verify`、`npm test` 表达审查与验证步骤。这些命令取决于实际版本、已安装技能及项目脚本，本篇不据此认定当前项目都具备，也不虚构测试通过记录。

验证并确认改动范围后，再执行 `git add`、检查暂存差异、`git commit`。提交说明写本次实际新增行为，而不是原样复制讲义的 `feat: new login flow`。

最后压缩对话时，保留：本次需求、已选方案、已修改文件、实际验证结果、提交标识和未完成事项。这样下一轮既有可恢复的代码版本，也有清楚的任务状态。

## 15. 重新进入会话：恢复讨论不会自动恢复代码

讲义还列出了会话恢复命令。当前官方 CLI 参考中的常用写法是：

在项目目录的**终端**中执行：

```bash
claude --continue
```

继续当前目录最近的会话。若要选择会话：

```bash
claude --resume
```

若已经知道会话 ID 或名称，可以传给 `--resume`：

```bash
claude --resume "accounting-export"
```

这里的名称只是示例，实际会话需要已存在。当前文档还支持用 `--name` 设置显示名称。

讲义中的 `claude --list-sessions`、`claude --session "debug-auth-issue"` 未出现在本次核对的官方 CLI 参数表中，本篇不作为当前通用命令推荐。注意 `--session-id` 是另一项参数，不等于任意会话名称。

恢复会话后，先确认当前目标、仓库状态和最近提交。文件可能已经被其他操作修改，旧对话里的“当前代码”不一定仍是磁盘上的当前代码。

## 16. /rewind：选择一个检查点并决定恢复什么

视频 13 介绍 `/rewind`。在支持该功能的 Claude Code 会话输入框中执行：

```text
/rewind
```

当前官方文档也支持在**输入框为空时连续按两次 Esc**打开回退菜单；若输入框有文本，双 Esc 的行为可能是先清空输入。

Claude Code 会为开启新一轮的用户提示建立检查点，并跟踪自身文件编辑工具产生的改动。选择目标检查点后，关键恢复选项有：

| 选项 | 文件 | 对话 |
|---|---|---|
| `Restore code and conversation` | 恢复受跟踪的文件改动 | 恢复到所选位置 |
| `Restore conversation` | 保持当前文件 | 恢复到所选位置 |
| `Restore code` | 恢复受跟踪的文件改动 | 保留当前对话 |

只有该检查点后存在可恢复的受跟踪文件改动时，菜单才提供相应的代码恢复选项。不能看到一条历史消息就认为一定能恢复任意文件。

**补充：**当前菜单还支持从指定位置总结一部分对话，以释放上下文。总结操作不是文件回退，不能把它和 `Restore code` 混为一件事。

操作前明确自己需要的是“重新讨论方案”“撤销实现”还是“两者一起回到原点”。只恢复对话后，当前文件仍然存在；只恢复文件后，对话中可能仍提到已经撤销的实现，需要把新状态告诉 Claude。

## 17. 检查点能回退哪些改动

配套讲义提到“撤销你刚才的改动”“把某文件恢复到改之前”，并列出内部备份与文件历史目录。自然语言请求会让 AI 尝试反向修改；它与选择已记录的检查点不同，仍要检查实际差异。

当前官方检查点文档明确了以下边界：

| 改动来源 | `/rewind` 的边界 |
|---|---|
| Claude 的文件编辑工具 | 属于主要跟踪对象 |
| Bash/Shell 命令写入、移动或删除文件 | 不由检查点自动跟踪 |
| 用户手工修改、其他并发会话修改 | 不能依赖当前会话完整捕获 |
| 安装依赖、执行程序、数据库写入 | 不等于受跟踪的代码编辑，不能依赖回退撤销所有影响 |

对记账项目而言，恢复一个 TypeScript 文件不代表已经录入的账单被撤销，也不代表 `npm install` 或打包生成的全部产物已恢复。

讲义所列 `~/.claude/backups/`、`~/.claude/file-history/` 涉及内部文件与版本行为。本次核对的官方说明强调通过检查点恢复，不能只凭这些目录名称认定任意编辑都有可用备份。

检查点有保存数量与保留期限，适用于会话内快速恢复；长期版本历史继续使用 Git。不要把 `/rewind` 当成 Git 提交的替代品。

## 18. Git 回退：根据文件所处状态选择操作

讲义按未暂存、已暂存、已提交、已推送区分回退。下面保留关键命令并解释影响；这些都是教学示例，**本次没有执行回退**。

### 未暂存：恢复指定文件

在项目仓库根目录，假设本次要撤销的文件是 `src/App.tsx`，先查看差异，再决定是否放弃：

```bash
git diff -- src/App.tsx
git restore -- src/App.tsx
```

默认 `git restore` 从暂存区恢复工作区中的已跟踪文件。它会丢弃该文件尚未暂存的修改。讲义旧写法是 `git checkout -- <file>`，也展示了作用于整个目录的 `git restore .`；需要局部恢复时应明确路径。

### 已暂存：取消暂存与丢弃修改是两步

现代写法：

```bash
git restore --staged -- src/App.tsx
```

它取消暂存，文件修改仍留在工作区。讲义对应的写法为 `git reset HEAD <file>`。

只有确认还要丢弃工作区修改时，才继续执行 `git restore -- src/App.tsx`。不能把“取消暂存”理解为“文件已经恢复”。

### 已提交：先决定是否保留改动

| 讲义命令 | 实际作用 |
|---|---|
| `git reset --soft HEAD~1` | 当前分支退回一个提交，改动保留在暂存区，工作区内容不变 |
| `git reset --hard HEAD~1` | 当前分支、暂存区和受影响的已跟踪文件回到上一个提交，未保存改动可能丢失 |
| `git revert HEAD` | 创建一个反向提交，抵消指定提交的影响，保留原有历史 |

讲义将 `--soft` 概括为“保留改动在 working tree”，这里补充明确：**改动仍处于已暂存状态**。

对已经共享的提交，讲义推荐 `git revert`，避免直接改写共同历史。它也可能产生冲突，处理后仍需验证。讲义同时列出的强制推送回退会改写远端历史，不属于本篇本地迭代的执行步骤。

## 19. Stash 和分支：用途不同于代码快照

讲义还给出两个检查点相关示例：

```bash
git stash push -m "checkpoint: before claude changes"
git checkout -b experiment/claude-refactor
```

第一条把当前部分未提交改动暂时收起；第二条创建并切换分支。不能把这两条都理解成“当前所有文件已经永久保存”。

**补充：**默认 `git stash push` 不包含未跟踪文件。如果临时修改包含新文件，先检查 `git status`，需要纳入时使用 `-u`：

```bash
git stash push -u -m "checkpoint: before claude changes"
git stash list
```

恢复某一项时，可先查看内容再应用：

```bash
git stash show -p "stash@{0}"
git stash apply "stash@{0}"
```

`apply` 后原 stash 仍保留，恢复时也可能有冲突。引号让 `stash@{0}` 在 PowerShell 中也能作为原样参数传入。

创建分支只建立并切换一个分支引用，不会自动把工作区和未跟踪文件变成新提交。要获得明确的代码基线，仍需提交；要暂时移开修改，使用合适的 stash。

## 20. 恢复以后，重新确认文件和行为

无论使用 `/rewind` 还是 Git 回退，完成后都应确认磁盘状态与当前任务一致：

```bash
git status
git diff
git diff --cached
git log -3 --oneline
```

`/rewind` 不负责把 Git 分支指针一并退回旧提交，因此检查点恢复后可能在当前提交上产生新的工作区差异。这并不矛盾：文件恢复与版本历史操作是两套机制。

对记账项目，接着按被影响的范围检查前端构建和对应功能。若只恢复了导出模块，至少确认导出数据与当前账单一致；若恢复了配置或依赖声明，需要同时核对环境与构建结果。

把新的状态说明给 Claude：哪些实现已撤销、哪些文件和数据仍保留、现在继续哪一个目标。需要保存当前结果时，检查差异后再创建新的 Git 提交。

本篇的操作顺序由此形成：**保存代码基线 → 管理上下文与长期约定 → 实现并验证一个功能 → 再次存档 → 必要时选择恰当的恢复方式。**

## 21. 核对来源与实践状态

课程来源：

- [VibeCoding实战网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)：已核对视频 10—13 的完整名称。
- [Part2 工程化控制和Agent开发](https://www.yuque.com/xxcls/vibecoding/ia0yu2m1v3sl537p)：已读取正文及原始代码块；本篇使用上下文、Memory、代码回退和相关功能迭代说明。

用于就地核对差异的官方资料：

- [项目说明与自动记忆](https://code.claude.com/docs/en/memory)：`CLAUDE.md`、`/init`、`/memory`、自动记忆路径与加载方式。
- [Checkpointing](https://code.claude.com/docs/en/checkpointing)：`/rewind`、恢复选项及跟踪边界。
- [CLI reference](https://code.claude.com/docs/en/cli-reference)：继续和恢复会话的参数。
- [Best practices](https://code.claude.com/docs/en/best-practices)：`/compact` 指定重点、任务间清理上下文及验证流程。

本次只新增本篇 Notebook。没有在当前仓库执行初始化、暂存、提交、stash、分支创建、文件回退或强制推送，也没有执行 Claude Code 命令和记账项目构建。

配套讲义相关部分以文字、表格和代码为主，没有需保留的原始教学插图；本篇不添加装饰图。